Installations

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn openai

Imports

In [2]:
import pandas as pd
import numpy as np
from openai import OpenAI
import joblib
import importlib
import matplotlib.pyplot as plt
importlib.reload(plt)
# import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [3]:
df=pd.read_csv("student_performance.csv")
df.head()
df.info()
df.describe()

FileNotFoundError: [Errno 2] No such file or directory: 'student_performance.csv'

Data Preprocessing

In [ ]:
df.isnull().sum()

In [ ]:
num_cols=df.select_dtypes(include=np.number).columns
for col in num_cols:
    df[col]=df[col].fillna(df[col].median())

In [ ]:
cat_cols=df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col]=df[col].fillna(df[col].mode()[0],)

Encoding

In [ ]:
df.select_dtypes(include="object").columns
le=LabelEncoder()
for col in cat_cols:
    df[col]=le.fit_transform(df[col])

Feature Selection

In [ ]:
features=["StudyHours","Attendance","Resources","Extracurricular","Motivation","LearningStyle",
          "OnlineCourses","Discussions","AssignmentCompletion","EduTech","StressLevel"]


Feature Scaling

In [ ]:
X=df[features]
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

Finding Best K

In [ ]:
inertia=[]
for k in range(2,11):
    kmeans=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=kmeans.fit_predict(X_scaled)
    score=silhouette_score(X_scaled,labels)
    print(f"K={k}, Silhouette Score={score:.4f}")
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(2,11),inertia,marker='o')
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

Training K Means

In [ ]:
kmeans=KMeans(n_clusters=5,random_state=42,n_init=10)
df["Cluster"]=kmeans.fit_predict(X_scaled)
df["Cluster"].value_counts().sort_index()

In [ ]:
cluster_summary = df.groupby("Cluster")[features].mean(numeric_only=True)
cluster_summary

In [ ]:
df.groupby("Cluster")[["ExamScore","FinalGrade"]].mean()

Evaluation

In [ ]:
score=silhouette_score(X_scaled,df["Cluster"])
print(score)

Analysing Clusters

In [ ]:
persona_types={0:"Self Learner",
               1:"Social Learner",
               2:"Silent Learner",
               3:"Balanced Learner",
               4:"Elite Learner"}
df["Persona"]=df["Cluster"].map(persona_types)

Visualization

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x="Persona",data=df)
plt.xlabel("Persona Type")
plt.ylabel("No. of Students")
plt.xticks(rotation=20)
plt.title("Distribution of Learners Persona")
plt.show()

In [ ]:
persona_scores = df.groupby("Persona")["ExamScore"].mean()
plt.figure(figsize=(8,5))
persona_scores.plot(kind="bar")
plt.title("Average Exam Score by Learner")
plt.xlabel("Persona Type")
plt.ylabel("Average Exam Score")
plt.xticks(rotation=20)
plt.show()

In [ ]:
persona_stress = df.groupby("Persona")["StressLevel"].mean()
plt.figure(figsize=(8,5))
persona_stress.plot(kind="bar")
plt.title("Average Stress Level by Learner")
plt.xlabel("Persona Type")
plt.ylabel("Average Stress Level")
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    df[features + ["ExamScore"]].corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
persona_text = {"Self Learner":"Independent and self-motivated learner who mostly studies individually.",
                "Social Learner":"Learns the most by collaboration and discussions.",
                "Silent Learner":"Learns mostly by observing, having limited collaborations and being quiet.",
                "Balanced Learner":"Balances self study and collaborations effectively.",
                "Elite Learner":"Highly focused learner with strong performance and discipline."
}

AI INTEGRATION FOR PERSONALIZED PATH GENERATION

In [ ]:
client=OpenAI(api_key='YOUR_GROQ_API_KEY',base_url="https://api.groq.com/openai/v1")

In [ ]:
def pathway_gen(student):
    persona=student["Persona"]
    prompt=f"""
    You are an AI educational mentor and these are the details of a student :-
    Persona = {persona}
    Persona Text = {persona_text[persona]}
    Student's details :- 
    	StudyHours={student['StudyHours']},
        Attendance={student['Attendance']},
        Motivation={student['Motivation']},
        LearningStyle={student['LearningStyle']},
        OnlineCourses={student['OnlineCourses']},
        AssignmentCompletion={student['AssignmentCompletion']}
        StressLevel={student['StressLevel']}
        Motivation Levels:
        0 = Low
        1 = Medium
        2 = High
        Motivation Levels:
        0 = Low
        1 = Medium
        2 = High
    Generate Profile summary, Weaknesses, Strengths, Areas to work on, Resources, a personalized learning pathway
    And generate all this information in a structured way.
    """

    response=client.chat.completions.create(model="llama-3.3-70b-versatile",messages=[{"role":"user","content":prompt}])
    return response.choices[0].message.content

In [ ]:
student = df.iloc[0]

pathway = pathway_gen(student)

print(pathway)